In [4]:
import feedparser 
import requests
import re
import pandas as pd

In [5]:
def get_RSS(url):
    return feedparser.parse(url)

In [6]:
def get_new_infos(url):
    '''
    Renvoie une liste de item
    '''
    RSS_page = get_RSS(url)
    list_new = []
    for i in range(len(RSS_page.entries)):
        item = RSS_page.entries[-1-i]
        if item in DATAFRAME:
            print("L'item existe déjà dans la dataframe.")
            break
        else:
            list_new.append(item)
    return list_new

In [7]:
def get_CVE_list(url):
    response = requests.get(url + "json/") 
    data = response.json()

    cve_pattern = r"CVE-\d{4}-\d{4,7}" 
    cve_list = list(set(re.findall(cve_pattern, str(data))))

    return cve_list

In [8]:
def get_ID_ANSSI_from_link(url):
    if url[30] == "v":
        return url[34:54]
    else:
        return url[36:55]

In [ ]:
def Use_API_CVE(cve_ID):
    '''
    Renvoie un dictionnaire correspondant au CVE en question contenant:
    - la description
    - le score CVSS
    - le type CWE
    - la description du type CWE
    - la liste des produits affectés. Chaque élément de la liste est un dictionnaire contenant le nom du produit, le vendeur, et une liste des versions affectées.
    '''
    url = f"https://cveawg.mitre.org/api/cve/{cve_ID}" 
    response = requests.get(url) 
    data = response.json()

    CVE_dico = {}

    CVE_dico["description"] = data["containers"]["cna"]["descriptions"][0]["value"]

    #ATTENTION tous les CVE ne contiennent pas nécessairement ce champ, gérez l’exception,  
    #ou peut etre au lieu de cvssV3_0 c’est cvssV3_1 ou autre clé 

    try:
        CVE_dico["cvss"] = data["containers"]["cna"]["metrics"][0]["cvssV3_1"]["baseScore"]  
    except:
        CVE_dico["cvss"] = "Non disponible"
    

    CVE_dico["cwe"] = "Non disponible"
    CVE_dico["cwe_desc"] = "Non disponible"


    problemtype = data["containers"]["cna"].get("problemTypes", {})
    if problemtype and "descriptions" in problemtype[0]:
        CVE_dico["cwe"] = problemtype[0]["descriptions"][0].get("cweId", "Non disponible")
        CVE_dico["cwe_desc"] = problemtype[0]["descriptions"][0].get("description", "Non disponible")

    # Extraire les produits affectés
    i=0
    product_list = []
    affected = data["containers"]["cna"]["affected"]
    for product in affected:
        product_list.append({})
        product_list[i]["product"] = product["product"]
        product_list[i]["vendor"] = product["vendor"]
        product_list[i]["versions"] = [v["version"] for v in product["versions"] if v["status"] == "affected"]
        i += 1

    CVE_dico["affected_products"] = product_list

    return CVE_dico

In [ ]:
def use_API_EPSS(cve_ID):
    url = f"https://api.first.org/data/v1/epss?cve={cve_ID}" 
    # Requête GET pour récupérer les données JSON 
    response = requests.get(url) 
    data = response.json() 
    # Extraire le score EPSS 
    epss_data = data.get("data", []) 
    if epss_data: 
        return epss_data[0]["epss"]
    else: 
        return "Non disponible"